<a href="https://colab.research.google.com/github/Whizz-tamie/gnn-indaba-2026/blob/main/notebooks/GNN_Tutorial_Live.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Graph Neural Networks: Foundations and Applications for Real-World Networks
### Live Session Notebook — Deep Learning Indaba 2026: Sovereign Intelligence, African Scale

*Prepared by Godbless James*

This notebook takes you from raw mobile money transaction records to a trained Graph
Neural Network, compared against a simple non-graph baseline.

**Task**: flag mobile money accounts likely to be involved in fraud, using a subsampled,
prepared slice of the [MoMTSim synthetic mobile money transaction dataset](https://data.mendeley.com/datasets/zhj366m53p/2)
(Azamuke, Katarahweire & Bainomugisha, Makerere University — CC BY 4.0).

**By the end of this notebook you will have:**
- Turned a transaction table into a graph (nodes, edges, features, labels)
- Trained a small Graph Convolutional Network (GCN) for account-level fraud detection
- Compared it against a simple MLP trained on the *same* node features, to see what the graph structure actually buys you

## 1. Setup

In [1]:
!pip install -q torch torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.2 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected
from sklearn.metrics import precision_score, recall_score, roc_auc_score, average_precision_score

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch: {torch.__version__}")

PyTorch: 2.11.0+cpu


## 2. Load the prepared sample

The full MoMTSim dataset has 1.72 million transactions. To keep the graph compact enough to train on quickly, this notebook uses a small, pre-prepared sample instead — around 50k transactions from the most active accounts, already subsampled for a dense enough neighbourhood structure. It downloads automatically below, so there's no manual upload needed.

In [3]:
!wget -q "https://raw.githubusercontent.com/Whizz-tamie/gnn-indaba-2026/refs/heads/main/data/momtsim_sample.csv" -O momtsim_sample.csv
df = pd.read_csv('momtsim_sample.csv')
df['initiator'] = df['initiator'].astype(str)
df['recipient'] = df['recipient'].astype(str)
print(f"Shape: {df.shape}")
df.head()

Shape: (50000, 10)


,step,transactionType,amount,initiator,oldBalInitiator,newBalInitiator,recipient,oldBalRecipient,newBalRecipient,isFraud
0,65,TRANSFER,18121.98,4251822106706232,4859621.86,4841499.88,4037533641984801,23795.04,41917.02,0
1,41,DEPOSIT,105462.76,4409271508181738,4505505.11,4610967.87,61-0007655,97397.35,97397.35,0
2,127,TRANSFER,39246.31,4360579409293775,1418990.65,1379744.34,4904942044067710,134020.81,173267.12,1
3,29,TRANSFER,42289.90,4210469501637180,2671271.06,2628981.16,4167790825993378,42359.92,84649.82,0
4,124,TRANSFER,17259.03,4197201698266276,3586965.40,3569706.37,4275853117454845,98396.93,115655.95,0


## 3. From transaction table to graph

A transaction table has one row per transaction. A graph has one node per **account**
and one edge per **transaction**. Four steps get us there.

**Step 1 — assign each account an integer node index.**

In [4]:
all_accounts = pd.Index(pd.concat([df['initiator'], df['recipient']]).unique())
account_to_idx = {acc: i for i, acc in enumerate(all_accounts)}
n_nodes = len(all_accounts)
print(f"Number of accounts (nodes): {n_nodes:,}")

Number of accounts (nodes): 13,237


**Step 2 — build `edge_index`.**

PyTorch Geometric aggregates information *into* each node from its neighbours along the
edges pointing at it. Our transactions are naturally directed (initiator → recipient),
but the accounts we're trying to classify are the *initiators* — if we leave the graph
directed, a labelled account only receives messages when it happens to be someone else's
recipient, which throws away most of the signal about who it paid. For this introductory
example we convert to an **undirected** graph with `to_undirected`, so every account
aggregates information about everyone it transacted with, regardless of direction.


In [5]:
src = df['initiator'].map(account_to_idx).to_numpy()
dst = df['recipient'].map(account_to_idx).to_numpy()
edge_index = torch.tensor(np.stack([src, dst]), dtype=torch.long)
edge_index = to_undirected(edge_index)
print(f"edge_index shape: {tuple(edge_index.shape)}  (2 x num_edges, undirected)")
print(f"Average degree: {edge_index.shape[1] / n_nodes:.2f}")

edge_index shape: (2, 92048)  (2 x num_edges, undirected)
Average degree: 6.95


**Step 3 — build node features.**

Each account gets a feature vector summarising its transaction behaviour: how much it sent/received, how many transactions, and its net flow. The GNN combines this with graph structure; the MLP baseline below sees *only* this — no structure at all.

In [6]:
sent = df.groupby('initiator').agg(out_count=('amount', 'size'), out_sum=('amount', 'sum'),
                                    out_mean=('amount', 'mean')).reindex(all_accounts).fillna(0)
recv = df.groupby('recipient').agg(in_count=('amount', 'size'), in_sum=('amount', 'sum'),
                                    in_mean=('amount', 'mean')).reindex(all_accounts).fillna(0)
feat = pd.concat([sent, recv], axis=1)
feat['tx_total'] = feat['out_count'] + feat['in_count']
feat['net_flow'] = feat['out_sum'] - feat['in_sum']
x = torch.tensor(feat.to_numpy(), dtype=torch.float32)
x = (x - x.mean(0)) / (x.std(0) + 1e-6)  # standardise
print(f"Node feature matrix: {tuple(x.shape)}")

Node feature matrix: (13237, 8)


**Step 4 — build labels and train/val/test masks.**

An account is labelled fraudulent if it ever *initiated* a fraudulent transaction — fraud
in this dataset originates from the sender side, so labelling recipients would mean
labelling victims, which isn't what we want.

In [7]:
fraud_accounts = set(df.loc[df['isFraud'] == 1, 'initiator'].unique())
y = torch.tensor([1 if acc in fraud_accounts else 0 for acc in all_accounts], dtype=torch.long)
print(f"Fraudulent accounts: {int(y.sum())} of {n_nodes} ({y.float().mean()*100:.2f}%)")

g = torch.Generator().manual_seed(42)
perm = torch.randperm(n_nodes, generator=g)
n_train, n_val = int(0.6 * n_nodes), int(0.2 * n_nodes)
train_mask = torch.zeros(n_nodes, dtype=torch.bool); train_mask[perm[:n_train]] = True
val_mask   = torch.zeros(n_nodes, dtype=torch.bool); val_mask[perm[n_train:n_train+n_val]] = True
test_mask  = torch.zeros(n_nodes, dtype=torch.bool); test_mask[perm[n_train+n_val:]] = True

data = Data(x=x, edge_index=edge_index, y=y,
            train_mask=train_mask, val_mask=val_mask, test_mask=test_mask)
print(data)

Fraudulent accounts: 2317 of 13237 (17.50%)
Data(x=[13237, 8], edge_index=[2, 92048], y=[13237], train_mask=[13237], val_mask=[13237], test_mask=[13237])


## 4. Baseline: train an MLP on the account features

Before reaching for a GNN, it's worth asking a more basic question: how far do the node
features alone get you, with no notion of graph structure at all? We start with a simple
MLP trained on the same node features built in Section 3. Whatever score it gets becomes
the number the GCN in Section 5 has to beat.

Fraud is a minority class (roughly 17-18% of accounts here), so we weight the loss
inversely to class frequency and report AUC / average precision (AP) rather than
accuracy, which would be misleading on imbalanced data.

In [8]:
n_pos = int(y[train_mask].sum())
n_neg = int(train_mask.sum()) - n_pos
class_weights = torch.tensor([1.0, n_neg / max(n_pos, 1)], dtype=torch.float32)

def train_model(model, data, epochs=80, lr=0.01, wd=5e-4):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    for epoch in range(1, epochs + 1):
        model.train()
        opt.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask], weight=class_weights)
        loss.backward()
        opt.step()
    return model

def evaluate(model, data):
    model.eval()
    with torch.no_grad():
        logits = model(data.x, data.edge_index)
        probs = F.softmax(logits, dim=1)[:, 1].numpy()
        preds = logits.argmax(dim=1).numpy()
    y_true = data.y[data.test_mask].numpy()
    y_pred = preds[data.test_mask.numpy()]
    y_prob = probs[data.test_mask.numpy()]
    return {
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall':    recall_score(y_true, y_pred, zero_division=0),
        'AUC':       roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else float('nan'),
        'AP':        average_precision_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else float('nan'),
    }

In [9]:
class MLP(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.3):
        super().__init__()
        self.fc1 = torch.nn.Linear(in_dim, hidden_dim)
        self.fc2 = torch.nn.Linear(hidden_dim, out_dim)
        self.dropout = dropout

    def forward(self, x, edge_index=None):  # edge_index accepted but ignored — no structure used
        x = F.relu(self.fc1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.fc2(x)

print('Training MLP baseline...')
mlp = MLP(in_dim=x.shape[1], hidden_dim=32, out_dim=2)
mlp = train_model(mlp, data)
mlp_metrics = evaluate(mlp, data)
print(f"MLP test metrics: {mlp_metrics}")

Training MLP baseline...
MLP test metrics: {'precision': 0.7865748709122203, 'recall': 0.9956427015250545, 'AUC': np.float64(0.9864643080723482), 'AP': np.float64(0.9237239141912066)}


## 5. Train a GCN — does structure beat the baseline?

Now the GNN. About 20 lines: two `GCNConv` layers with a ReLU and dropout between them.
Same features as the MLP, same training setup, same evaluation — the only difference is
that `GCNConv` also uses `edge_index`, so each account's representation is updated using
its neighbours' features too, not just its own.

In [10]:
class GCN(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.3):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, out_dim)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index)

print('Training GCN...')
gcn = GCN(in_dim=x.shape[1], hidden_dim=32, out_dim=2)
gcn = train_model(gcn, data)
gcn_metrics = evaluate(gcn, data)
print(f"GCN test metrics: {gcn_metrics}")

Training GCN...
GCN test metrics: {'precision': 0.7889273356401384, 'recall': 0.9934640522875817, 'AUC': np.float64(0.9856342516703143), 'AP': np.float64(0.9167873990627998)}


**Comparing the two.**

In [11]:
summary = pd.DataFrame({'MLP (no structure)': mlp_metrics, 'GCN (with structure)': gcn_metrics}).T
summary.round(3)

,precision,recall,AUC,AP
MLP (no structure),0.787,0.996,0.986,0.924
GCN (with structure),0.789,0.993,0.986,0.917


**Reading this table honestly.** On this dataset, the two scores land close together —
the GCN doesn't clearly beat the MLP. That's a real result worth sitting with rather than
explaining away: it means the node features we built in Section 3 (transaction counts,
sums, means) already carry most of the signal on their own, because they're aggregated
from the very same transactions that produced the fraud label in the first place. Adding
graph structure on top of features that already leak the answer has little left to prove.

That raises the obvious next question — if the features are doing all the work here, is
the graph structure contributing *anything at all*? Section 6 answers that directly, by
taking the features away.

## 6. Does the graph structure matter on its own?

Section 5 left an open question: the features already carry most of the signal, so the comparison there couldn't tell us whether the graph structure is doing anything at all.
This section isolates that directly. We give the GCN and the MLP node features that carry **no behavioural information whatsoever** — small random noise, nothing derived from transactions — and see how each model does. If the GCN still performs well here, that performance can only be coming from the graph structure itself: which accounts are connected to which.

In [12]:
x_noise = torch.randn(n_nodes, 4) * 0.01
data_noise = Data(x=x_noise, edge_index=edge_index, y=y,
                   train_mask=train_mask, val_mask=val_mask, test_mask=test_mask)

print('Training MLP on uninformative features...')
mlp_noise = MLP(in_dim=4, hidden_dim=32, out_dim=2)
mlp_noise = train_model(mlp_noise, data_noise)
mlp_noise_metrics = evaluate(mlp_noise, data_noise)
print(f"MLP (noise features) test metrics: {mlp_noise_metrics}")

print('Training GCN on uninformative features...')
gcn_noise = GCN(in_dim=4, hidden_dim=32, out_dim=2)
gcn_noise = train_model(gcn_noise, data_noise)
gcn_noise_metrics = evaluate(gcn_noise, data_noise)
print(f"GCN (noise features) test metrics: {gcn_noise_metrics}")

Training MLP on uninformative features...
MLP (noise features) test metrics: {'precision': 0.17333836858006044, 'recall': 1.0, 'AUC': np.float64(0.4988395134714969), 'AP': np.float64(0.17164224660187088)}
Training GCN on uninformative features...
GCN (noise features) test metrics: {'precision': 0.7399030694668821, 'recall': 0.9978213507625272, 'AUC': np.float64(0.9877880191211553), 'AP': np.float64(0.9286593360802531)}


In [13]:
ablation = pd.DataFrame({
    'MLP (noise features)': mlp_noise_metrics,
    'GCN (noise features)': gcn_noise_metrics,
}).T
ablation.round(3)

,precision,recall,AUC,AP
MLP (noise features),0.173,1.000,0.499,0.172
GCN (noise features),0.740,0.998,0.988,0.929


**This is the gap that matters.** With no informative features at all, the MLP has nothing to work with and lands near chance (AUC around 0.5). The GCN, given the exact same uninformative features, still performs well — because it isn't relying on the features at all here. It's reading fraud signal directly from the graph's connectivity: which accounts share neighbourhoods with which other accounts. That gap is the clearest evidence in this notebook that relational structure — not just node-level behaviour — carries real information about fraud in this dataset.

**A note on all the scores in this notebook.** Every result here — the baseline, the GCN, and this ablation — comes from MoMTSim, a *synthetic*, simulation-generated dataset with fraud patterns and features that are cleaner than anything you'd see in real transaction data. Treat these numbers as an **educational demonstration** of how message passing and relational structure can help, not as evidence that either model is ready for real-world fraud detection. A production system would need messier real data, temporal train/test splits (fraud evolves over time), and testing against fraud typologies the
model has never seen — none of which this notebook attempts.

## 7. Want to go further?

This notebook stopped at a baseline-vs-GCN comparison, plus the structure-only ablation,
on purpose — a small number of ideas covered clearly. A companion [take-home
notebook](./GNN_Tutorial_TakeHome.ipynb) picks up from here and covers:

- Full exploratory analysis of the MoMTSim dataset and the subsampling rationale
- A **Graph Attention Network (GAT)**, trained on this same task, with learned attention
  weights instead of the GCN's fixed neighbour weighting
- A small visualisation of the fraud neighbourhood structure
- Exercises, including building a **heterogeneous graph** that distinguishes clients,
  merchants, banks, and the fraudster role as different node/edge types
- A curated further-reading list

**Takeaways.** Transaction records become a graph through four concrete steps (node index, edge_index, features, labels). A non-graph baseline is worth training first, not as an afterthought — here, it revealed that the node features alone already carry most of the fraud signal, which is itself a useful, honest finding. And when that signal was stripped away entirely, the GCN kept performing while the baseline collapsed to chance — showing that the graph's structure carries real information about fraud on its own, independent of any node-level features at all.